# 프롬프트 → 순수 text-to-image → 오류 지도 → 재역확산

첫 생성에는 이미지·폰트·표준 글자·획 모델을 전달하지 않습니다. 입력은 프롬프트와 seed뿐입니다.
1 준비 → 2 프롬프트로 실제 생성 → 3 저장 모델/문자 조건 선택 → 4 생성 사진의 문자 영역과 오류 지도 → 5 재역확산 → 6 비교.
실제 문자가 틀리거나 배치가 달라도 폰트 렌더링으로 바꾸거나 결과를 골라내지 않습니다.
문자 위치를 고정 2×2로 가정하지 않습니다. 생성 결과에서 각 글자의 bbox를 지정해야 합니다.
현재 오류 지도 보정은 밝은 단색 배경의 추가 잉크 삭제용입니다. 사진 배경이 복잡하면 명시적으로 중단하며, 폰트 이미지로 대체하지 않습니다.
생성과 보정 모두 Qwen-Image-2512를 사용합니다. 코랩 A100 40GB 이상을 선택하세요. 4비트 양자화로 로딩하며 T4로 자동 대체하지 않습니다.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile, json
root = Path(tempfile.mkdtemp(prefix='hsqm_diffusion_score_', dir='/content'))
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
    'https://github.com/yechan25/hsqm.git', str(root)], check=True)
commit = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/huggingface/diffusers.git@8c54b38fea3d624d23d0e43c0c96037f080a79f9', 'transformers>=4.57,<5', 'accelerate', 'bitsandbytes>=0.48', 'safetensors',
    'timm', 'pandas', 'numpy', 'tqdm', 'matplotlib', 'pillow', 'scipy',
    'scikit-image', 'pyyaml', 'fonttools', 'ipywidgets'], check=True)
sys.path.insert(0, str(root))
for key in list(sys.modules):
    if key.split('.')[0] in ('stroke_model', 'font_test', 'diffusion_model'):
        del sys.modules[key]
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Colab에서 GPU 런타임으로 바꿔주세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('실행 GitHub 커밋:', commit)
from google.colab import drive, output
drive.mount('/content/drive')
output.enable_custom_widget_manager()

In [ ]:
from datetime import datetime
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
from stroke_model.convert_reference_csv import existing_path
from diffusion_model.prompt_generation import generate_prompt_image
from diffusion_model.qwen_runtime import load_qwen_pipeline
from huggingface_hub import model_info
HSQM=existing_path('/content/drive/Shareddrives/2026 자율연구/HSQM')
MODEL_ID='Qwen/Qwen-Image-2512'
MODEL_REVISION=model_info(MODEL_ID).sha
CANVAS_SIZE=1024
PROMPT='A photograph of a white sign displaying the Korean text "가나다라" in black ink. Exactly these four Hangul characters in a single line. No other text.'
EXPECTED_TEXT='가나다라'  # User intention, not rendered into an input image
GENERATION_STEPS=50
GENERATION_SEED=100
NEGATIVE_PROMPT='extra text, multiple lines, columns, table, watermark, logo, border'
RUN_DIR=HSQM/'prompt_then_repair'/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
qwen_pipeline=load_qwen_pipeline(MODEL_ID,revision=MODEL_REVISION)
first=generate_prompt_image(RUN_DIR/'stage1',prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,model_id=MODEL_ID,revision=MODEL_REVISION,
    steps=GENERATION_STEPS,seed=GENERATION_SEED,size=CANVAS_SIZE,pipeline=qwen_pipeline)
FIRST_IMAGE=Path(first['generated_image'])
fig,ax=plt.subplots(figsize=(7,7))
with Image.open(FIRST_IMAGE) as im:ax.imshow(im.convert('RGB'))
ax.set_title('Actual prompt-only generation; pixel coordinates for character ROIs');plt.show()
print('First-stage image/template/font inputs:',first['input_image_used'],first['template_used'],first['font_rendering_used'])

In [ ]:
from stroke_model.convert_reference_csv import existing_path
from diffusion_model.score_pilot import training_catalog, generate_pilot, score_pilot, load_catalog_reference
from diffusion_model.pilot_data import prepare_pilot_data
from diffusion_model.multiglyph_pilot import generate_multiglyph_pilot
from IPython.display import display
import ipywidgets as widgets
HSQM = existing_path('/content/drive/Shareddrives/2026 자율연구/HSQM')
CHECKPOINT = None  # 폰트 실험에서 사용한 best.pt의 전체 경로를 넣으면 그 모델 사용
DATA_ROOT = HSQM/'dataset/automated_train_dataset'  # 기존 학습 ZIP/TAR가 있는 폴더
CSV_PATH = None  # None: 체크포인트 옆에 저장된 학습 CSV를 자동 사용
TRAIN_MANIFEST = None  # 기존 세션의 유효한 목록이 있을 때만 선택적으로 지정. 없어도 자동 재생성
REFERENCE_ONLY = True  # 시험에는 기준 글자/기준 획만 사용. 전체 학습 NPZ 변환 생략
SCORE_CONFIG = {}  # 폰트 시험에서 값을 바꿨다면 동일한 설정 입력
SOURCE_CONFIG = {}
GLYPH_COUNTS = [4]
FIXED_CELL_PIXELS = None  # None: 전체 크기 고정. 192: 글자당 크기 고정 비교
STRENGTHS = [.65]
SEEDS = [100]
STEPS = 100

from stroke_model.detected_extra_guidance import load_error_head,predict_extra_sheet,DetectedExtraGuidance
ERROR_CHECKPOINT=None  # 학습한 새 오류 탐지기 best.pt. None이면 최신 학습 결과를 선택
THRESHOLD=.5  # 미보정 진단 임계값. 3번에서 실제 지도를 확인하세요.
KEEP_WEIGHT=20.
REPAIR_STEPS=50
REPAIR_STRENGTH=.3
REPAIR_SEED=101
if ERROR_CHECKPOINT is None:
    candidates=sorted((HSQM/'error_localizer_runs').glob('*/best.pt'))
    if not candidates:raise FileNotFoundError('ERROR_CHECKPOINT에 이미 학습한 새 오류 탐지기 best.pt 경로를 넣으세요. 이 노트북에서는 학습하지 않습니다.')
    ERROR_CHECKPOINT=candidates[-1]
# Read the detector metadata to use its EXACT frozen extractor, not an unrelated latest one.
info=torch.load(ERROR_CHECKPOINT,map_location='cpu',weights_only=True)
recorded=info.get('metadata',{}).get('extractor_checkpoint')
if CHECKPOINT is None and recorded:CHECKPOINT=existing_path(recorded)
if CHECKPOINT is None:raise ValueError('CHECKPOINT에 학습 당시 사용한 획 추출기를 지정하세요.')
preparation=prepare_pilot_data(CHECKPOINT,DATA_ROOT,csv_path=CSV_PATH,manifest=TRAIN_MANIFEST,reference_only=True)
TRAIN_MANIFEST=Path(preparation['manifest'])
from stroke_model.infer_topology import load_topology
extractor,size,_=load_topology(CHECKPOINT,torch.device('cuda'))
head,detector_info=load_error_head(ERROR_CHECKPOINT,CHECKPOINT,'cuda')
if FIRST_IMAGE is None:raise ValueError('FIRST_IMAGE에 직접 프롬프트로 생성한 이미지 경로를 지정하세요. 과거 폰트 기반 파일을 자동 선택하지 않습니다.')
FIRST_IMAGE=existing_path(FIRST_IMAGE)
print('오류 탐지기:',ERROR_CHECKPOINT)
print('정확히 대응하는 획 추출기:',CHECKPOINT)
print('재역확산할 생성물:',FIRST_IMAGE)
print('threshold calibrated:',detector_info['threshold_calibrated'])
import io, math
import numpy as np
from PIL import Image, ImageDraw
catalog = training_catalog(TRAIN_MANIFEST)
by_row = {item['manifest_row']: item for item in catalog}
needed = max(GLYPH_COUNTS)
if len(catalog) < needed:
    raise ValueError(f'사용 가능한 학습 기준은 {len(catalog)}개입니다. GLYPH_COUNTS를 줄이세요.')
reference_select = widgets.SelectMultiple(
    options=[(f"train row {item['manifest_row']} | {item['char_id'] or '(그림 확인)'}", item['manifest_row']) for item in catalog],
    value=tuple(item['manifest_row'] for item in catalog[:needed]), rows=12,
    description='여러 글자:', layout=widgets.Layout(width='98%'))
preview = widgets.Image(format='png', width=500, height=500)
def update_preview(change=None):
    selected = list(reference_select.value)[:needed]
    side = math.ceil(math.sqrt(needed))
    sheet = Image.new('RGB', (160*side, 180*side), 'white')
    draw = ImageDraw.Draw(sheet)
    for index, row in enumerate(selected):
        reference,_ = load_catalog_reference(by_row[row])
        glyph = Image.fromarray(np.uint8((1-reference[0])*255)).convert('RGB')
        x, y = (index%side)*160, (index//side)*180
        sheet.paste(glyph.resize((160,160)), (x,y))
        draw.text((x+4,y+160), f'slot {index+1} | row {row}', fill='black')
    buffer=io.BytesIO(); sheet.save(buffer,format='PNG'); preview.value=buffer.getvalue()
reference_select.observe(update_preview,names='value'); update_preview()
display(widgets.VBox([reference_select,preview]))
print('정확히 4개를 선택하세요. 같은 4개로 기본/유도 경로를 비교합니다.')
print('문자명이 없는 경우 그림으로 확인하세요. 서로 다른 글자를 비교하려면 같은 글자의 폰트 변형을 중복 선택하지 마세요.')
import ipywidgets as widgets
glyph_box=widgets.Text(description='4문자:',placeholder='예: 가 걔 곁 괴 (선택 순서와 일치)')

from PIL import Image
import numpy as np
from diffusion_model.multiglyph_pilot import compose_sheet
from diffusion_model.score_pilot import training_catalog
characters_box=glyph_box
print('같은 4문자를 선택하고, 문자 칸에 생성물의 목표 문자를 읽기 순서대로 입력하세요.')
display(characters_box)
with Image.open(FIRST_IMAGE) as im:display(im.convert('RGB'))
characters_box.value=EXPECTED_TEXT
CHAR_BOXES=[]  # Fill [(x0,y0,x1,y1), ...] for the four intended characters, in reading order
print('사진에서 각 글자 bbox를 CHAR_BOXES에 지정하세요. 잘못 생성된 문자/위치도 숨기지 않습니다.')

In [ ]:
from datetime import datetime
from huggingface_hub import model_info
from diffusion_model.run_topo_guidance import generate_guided_pair
chars=characters_box.value.split()
if len(chars)==1 and len(chars[0])==4:chars=list(chars[0])
if len(chars)!=4:raise ValueError('예: 가 걔 곁 괴 — 4문자를 입력하세요.')
selected=list(reference_select.value)
if len(selected)!=4:raise ValueError('표준 획 조건을 정확히 4개 선택하세요.')
by_row={r['manifest_row']:r for r in training_catalog(TRAIN_MANIFEST)}
_,slots=compose_sheet([by_row[r] for r in selected],CANVAS_SIZE)
if len(CHAR_BOXES)!=4:raise ValueError('CHAR_BOXES에 실제 생성 사진 속 네 글자의 bbox를 지정하세요. 자동 2x2 영역은 사용하지 않습니다.')
for slot,box in zip(slots,CHAR_BOXES):
    if len(box)!=4 or any(type(v) is not int for v in box):raise ValueError('Integer bboxes required')
    x0,y0,x1,y1=box
    if not (0<=x0<x1<=CANVAS_SIZE and 0<=y0<y1<=CANVAS_SIZE):raise ValueError('BBox outside image')
    slot['bbox']=list(box)

with Image.open(FIRST_IMAGE) as im:
    if im.size!=(CANVAS_SIZE,CANVAS_SIZE):raise ValueError('생성 이미지 크기와 CANVAS_SIZE를 맞추세요.')
    array=np.asarray(im.convert('RGB'),np.float32)/255
import hashlib
SOURCE_SHA=hashlib.sha256(Path(FIRST_IMAGE).read_bytes()).hexdigest()
source_rgb=torch.from_numpy(array.copy()).permute(2,0,1)[None].cuda()
head.cuda();extractor.cuda()
probability=predict_extra_sheet(head,extractor,source_rgb,slots,chars)
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(array);axes[0].set_title('Generated input')
axes[1].imshow(probability[0,0].cpu(),cmap='magma',vmin=0,vmax=1);axes[1].set_title('Trained detector probability')
axes[2].imshow(array);axes[2].imshow((probability[0,0].cpu()>=THRESHOLD),cmap='Reds',alpha=.4,vmin=0,vmax=1);axes[2].set_title('Fixed edit mask')
for ax in axes:ax.axis('off')
plt.show()
fixed_objective=DetectedExtraGuidance(source_rgb,probability,slots,THRESHOLD,KEEP_WEIGHT)
print('이 오류 지도가 재역확산에 실제 사용됩니다. 잘못된 위치이면 threshold/선택을 확인한 뒤 실행하세요.')

In [ ]:
from diffusion_model.qwen_repair import repair_qwen_pair
if hashlib.sha256(Path(FIRST_IMAGE).read_bytes()).hexdigest()!=SOURCE_SHA:raise RuntimeError('입력 이미지가 바뀌었습니다. 4번 셀부터 실행하세요.')
extractor.cpu();head.cpu();torch.cuda.empty_cache()
OUTPUT_DIR=RUN_DIR/('repair_'+datetime.now().strftime('%H%M%S_%f'))
generation=repair_qwen_pair(qwen_pipeline,FIRST_IMAGE,OUTPUT_DIR,fixed_objective,
    prompt=PROMPT,negative_prompt=NEGATIVE_PROMPT,seed=REPAIR_SEED,
    steps=REPAIR_STEPS,strength=REPAIR_STRENGTH,step_size=.02,backtracks=5)
np.savez_compressed(OUTPUT_DIR/'detector_plan.npz',probability=probability.cpu().numpy(),mask=fixed_objective.mask.cpu().numpy(),target=fixed_objective.target.cpu().numpy())
(OUTPUT_DIR/'detector_info.json').write_text(json.dumps(dict(detector_info,checkpoint=str(ERROR_CHECKPOINT),threshold=THRESHOLD,source_sha256=SOURCE_SHA),indent=2))
(OUTPUT_DIR/'git_commit.txt').write_text(commit)
print('채택된 가이드 단계:',generation['guided_accepted_steps'])
print('Qwen flow 단계별 loss:',generation['history'])

In [ ]:
pages={p['method']:p for p in generation['pages']}
fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,(title,path) in zip(axes,[('First generation',FIRST_IMAGE),('Re-denoise only',pages['baseline']['input']),('Detector-guided reverse diffusion',pages['guided']['input'])]):
    with Image.open(path) as image:ax.imshow(image.convert('RGB'))
    ax.set_title(title);ax.axis('off')
fig.tight_layout();fig.savefig(OUTPUT_DIR/'comparison.png',dpi=140);plt.show()
snapshots=sorted(OUTPUT_DIR.glob('guided_step_*_after.png'))
if snapshots:
    fig,axes=plt.subplots(1,min(6,len(snapshots)),figsize=(18,3),squeeze=False)
    for ax,path in zip(axes[0],snapshots[-6:]):
        with Image.open(path) as image:ax.imshow(image.convert('RGB'))
        ax.set_title(path.stem);ax.axis('off')
    plt.show()
import zipfile
from google.colab import files
archive=RUN_DIR.parent/(RUN_DIR.name+'_prompt_repair.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in RUN_DIR.rglob('*'):
        if path.is_file():z.write(path,path.relative_to(RUN_DIR))
files.download(str(archive))